<a href="https://colab.research.google.com/github/Tao0807/mingw-w64.github.io/blob/main/d1257272_%E9%99%B3%E9%9B%8B%E6%BF%A4__week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第三節　動手做：讓文字變成向量
逢甲大學資訊工程學系 · 生成式 AI 實務

**組別：** ______　**組員：** ______________________

今天**不用寫程式**。每個任務都是一個表單：在右邊改文字或參數 → 按左邊的 ▶ → 看結果，把觀察寫在最下面的記錄表。

| 時間 | 做什麼 |
|---|---|
| 5 分鐘 | 準備環境（按一次 ▶） |
| 約 10 分鐘 | 任務 1：詞向量遊戲（相似詞、類比） |
| 約 10 分鐘 | 任務 2：句子相似度，先猜再看 |
| 約 10 分鐘 | 任務 3：語意搜尋 vs 關鍵字搜尋 |
| 最後 | 填記錄表、各組分享 |

**繳交**：這份 notebook（保留執行結果）。「檔案 → 下載 → 下載 .ipynb」。

## 準備環境
1. （建議）「**執行階段 → 變更執行階段類型**」選 **T4 GPU**。沒有 GPU 也能跑，只是慢一點。
2. 按下面這格的 ▶，會下載英文詞向量（約 128 MB）和中文句向量模型（約 95 MB），大約 2–3 分鐘。
3. 看到 **✓ 準備完成** 就可以開始。

In [ ]:
# @title ▶ 準備環境（按一次就好）
import subprocess, sys, logging, warnings
warnings.filterwarnings("ignore")
print("1/4 安裝套件 …")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gensim", "jieba", "sentence-transformers"])

import numpy as np, torch
import plotly.express as px
from IPython.display import display, Markdown, HTML
device = "cuda" if torch.cuda.is_available() else "cpu"
print("    使用裝置：", device)

print("2/4 下載英文詞向量 GloVe（約 128 MB）…")
import gensim.downloader as api
glove = api.load("glove-wiki-gigaword-100")

print("3/4 下載中文句向量模型 bge-small-zh（約 95 MB）…")
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-small-zh-v1.5", device=device)
def embed(texts):
    return model.encode(texts, normalize_embeddings=True)

print("4/4 建立搜尋資料 …")
FAQ = [   # 範例資料：校園常見問題（虛構，僅供練習）
    "資訊設備故障或電腦無法開機，請到系辦填寫報修單",
    "教室的投影機沒有畫面時，請聯絡系館管理員",
    "選課時間與加退選規定請參考教務處公告",
    "畢業前需修滿 128 學分，並通過英文檢定",
    "獎學金申請請於每學期開學後兩週內送件",
    "學生證遺失可至註冊組申請補發，需繳交工本費",
    "宿舍網路斷線請先重新開機，仍無法連線再通報宿網中心",
    "請假需在系統上申請，並附上相關證明文件",
    "實驗室開放時間為平日早上九點到晚上九點",
    "專題成果展每年五月舉行，請各組準備海報與展示",
    "校園 Wi-Fi 帳號與密碼和學校信箱相同",
    "圖書館可以借用筆記型電腦，借期為一週",
    "學餐營業時間為週一到週五，週末不營業",
    "畢業專題需由指導教授同意後才能修課",
    "停車證每學期開放申請一次，名額有限",
]
import jieba, logging
jieba.setLogLevel(logging.WARNING)
PUNCT = set("，。、！？；：「」（）,.!?;:()　 ")
def tokenize(text):
    """用 jieba 斷詞，去掉標點與空白"""
    return [w for w in jieba.cut(text) if w.strip() and w not in PUNCT]
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
X = tfidf.fit_transform(FAQ)
doc_emb = embed(FAQ)
QUERY_PREFIX = "为这个句子生成表示以用于检索相关文章："

def bar(score):
    """把 0–1 的分數畫成一條長條"""
    n = max(0, min(20, round(score * 20)))
    return "█" * n + "░" * (20 - n)

print("\n✓ 準備完成！往下做任務 1。")

1/4 安裝套件 …
    使用裝置： cuda
2/4 下載英文詞向量 GloVe（約 128 MB）…
[==================================================] 100.0% 128.1/128.1MB downloaded
3/4 下載中文句向量模型 bge-small-zh（約 95 MB）…


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/27.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

4/4 建立搜尋資料 …

✓ 準備完成！往下做任務 1。


## 任務 1　詞向量遊戲（英文）
GloVe 是用大量英文文章訓練出來的詞向量。只能輸入**英文小寫單字**。

**1A 找相似詞**：輸入一個詞，看它的「鄰居」是誰。

In [ ]:
# @title 1A：找相似詞
word = "soccer"  # @param ["coffee", "king", "taiwan", "python", "apple", "happy", "football"] {allow-input: true}
word = word.strip().lower()
if word not in glove.key_to_index:
    print(f"「{word}」不在詞彙表裡，請換一個英文單字")
else:
    for w, s in glove.most_similar(word, topn=8):
        print(f"{s:.3f}  {bar(s)}  {w}")

0.873  █████████████████░░░  football
0.800  ████████████████░░░░  basketball
0.769  ███████████████░░░░░  volleyball
0.744  ███████████████░░░░░  hockey
0.742  ███████████████░░░░░  league
0.731  ███████████████░░░░░  sports
0.730  ███████████████░░░░░  club
0.724  ██████████████░░░░░░  rugby


**1B 類比推理**：A 之於 B，就像 C 之於 **?**　（計算方式：B − A + C）

先在記錄表寫下你猜的答案，再按 ▶。

In [ ]:
# @title 1B：A 之於 B ＝ C 之於 ?
example = "自己輸入（用下面三格）"  # @param ["man : king = woman : ?", "france : paris = japan : ?", "walk : walked = swim : ?", "good : better = bad : ?", "man : doctor = woman : ?", "自己輸入（用下面三格）"]
A = "apple"  # @param {type:"string"}
B = "fruit"  # @param {type:"string"}
C = "banana"  # @param {type:"string"}

if example.startswith("自己"):
    a, b, c = A.strip().lower(), B.strip().lower(), C.strip().lower()
else:
    left, right = example.split("=")
    a, b = [x.strip() for x in left.split(":")]
    c = right.split(":")[0].strip()

missing = [w for w in (a, b, c) if w not in glove.key_to_index]
if missing:
    print("這些詞不在詞彙表裡：", missing)
else:
    print(f"{a} : {b} = {c} : ?\n")
    for w, s in glove.most_similar(positive=[b, c], negative=[a], topn=5):
        print(f"{s:.3f}  {bar(s)}  {w}")

apple : fruit = banana : ?

0.650  █████████████░░░░░░░  bananas
0.649  █████████████░░░░░░░  fruits
0.641  █████████████░░░░░░░  coconut
0.624  ████████████░░░░░░░░  edible
0.593  ████████████░░░░░░░░  citrus


## 任務 2　句子相似度：先猜再看
輸入兩句中文，模型會算出 0 到 1 的相似度（越接近 1 越像）。

**試試看**：從選單選一組，**先在記錄表寫下你猜的分數**，再按 ▶。也可以自己打兩句。

In [ ]:
# @title 任務 2：兩句話有多像？
pair = "一詞多義：小米發表新手機｜我早餐喝小米粥"  # @param ["換句話說：我喜歡這門課｜我很喜歡這堂課", "否定：我喜歡這門課｜我不喜歡這門課", "同主題：今天好熱｜颱風要來了", "無關：我喜歡這門課｜今天的午餐是便當", "數字：報名費是 500 元｜報名費是 5000 元", "一詞多義：小米發表新手機｜我早餐喝小米粥", "自己輸入（用下面兩格）"]
sentence_1 = "期中考考得很好"  # @param {type:"string"}
sentence_2 = "這次考試成績不錯"  # @param {type:"string"}

if pair.startswith("自己"):
    a, b = sentence_1, sentence_2
else:
    a, b = pair.split("：", 1)[1].split("｜")
ea, eb = embed([a, b])
score = float(ea @ eb)
display(Markdown(f"**句子 1**：{a}  \n**句子 2**：{b}"))
print(f"\n相似度 {score:.3f}  {bar(score)}")

**句子 1**：小米發表新手機  
**句子 2**：我早餐喝小米粥


相似度 0.510  ██████████░░░░░░░░░░


## 任務 3　語意搜尋 vs 關鍵字搜尋
下面是一份校園 FAQ（範例資料，共 15 筆）。輸入一個問題，左邊是**關鍵字搜尋**（TF-IDF），右邊是**向量搜尋**（句向量），各列出最像的 3 筆。

**試試看**：用**跟 FAQ 不同的說法**來問，例如「上網連不上」、「肚子餓了去哪吃」，看哪一邊找得到。

In [ ]:
# @title 先看看 FAQ 有哪些內容
for i, d in enumerate(FAQ, 1):
    print(f"{i:>2}. {d}")

 1. 資訊設備故障或電腦無法開機，請到系辦填寫報修單
 2. 教室的投影機沒有畫面時，請聯絡系館管理員
 3. 選課時間與加退選規定請參考教務處公告
 4. 畢業前需修滿 128 學分，並通過英文檢定
 5. 獎學金申請請於每學期開學後兩週內送件
 6. 學生證遺失可至註冊組申請補發，需繳交工本費
 7. 宿舍網路斷線請先重新開機，仍無法連線再通報宿網中心
 8. 請假需在系統上申請，並附上相關證明文件
 9. 實驗室開放時間為平日早上九點到晚上九點
10. 專題成果展每年五月舉行，請各組準備海報與展示
11. 校園 Wi-Fi 帳號與密碼和學校信箱相同
12. 圖書館可以借用筆記型電腦，借期為一週
13. 學餐營業時間為週一到週五，週末不營業
14. 畢業專題需由指導教授同意後才能修課
15. 停車證每學期開放申請一次，名額有限


In [ ]:
# @title 任務 3：同一個問題，兩種搜尋
query = "停車證"  # @param ["上網連不上", "電腦壞了要找誰", "肚子餓了可以去哪裡吃飯", "畢業要多少學分", "證件不見了怎麼辦", "要怎麼申請錢"] {allow-input: true}

kw = (tfidf.transform([query]) @ X.T).toarray()[0]
se = embed([QUERY_PREFIX + query])[0] @ doc_emb.T

def table(scores):
    rows = "".join(f"<tr><td style='padding:4px 8px'>{scores[i]:.2f}</td><td style='padding:4px 8px'>{FAQ[i]}</td></tr>"
                   for i in scores.argsort()[::-1][:3])
    return f"<table>{rows}</table>"

display(HTML(f"""<h4>🔎 {query}</h4>
<div style='display:flex;gap:24px'>
<div><b>關鍵字搜尋（TF-IDF）</b><br>斷詞：{' / '.join(tokenize(query))}{table(kw)}</div>
<div><b>向量搜尋（句向量）</b>{table(se)}</div></div>"""))

0.40,停車證每學期開放申請一次，名額有限
0.00,畢業專題需由指導教授同意後才能修課
0.00,學餐營業時間為週一到週五，週末不營業
0.64,停車證每學期開放申請一次，名額有限
0.44,學生證遺失可至註冊組申請補發，需繳交工本費
0.39,請假需在系統上申請，並附上相關證明文件


## 加分挑戰　畫一張自己的「意思地圖」
輸入一串中文詞或短句（用逗號分開），模型會把它們畫在 2D 地圖上。意思相近的會靠在一起嗎？A:會

In [ ]:
# @title 加分：意思地圖
items = "人, 男, 女, 電腦, 手機, 桌遊, 手遊, 信用卡, 現金, 假日, 周末, 平日, 年輕人, 老人, 青少年, 硬幣, 金融卡, 黑色, 白色, 透明, 紅色"  # @param {type:"string"}
from sklearn.decomposition import PCA
words = [w.strip() for w in items.replace("，", ",").split(",") if w.strip()]
if len(words) < 3:
    print("請至少輸入 3 個詞")
else:
    xy = PCA(n_components=2).fit_transform(embed(words))
    fig = px.scatter(x=xy[:, 0], y=xy[:, 1], text=words, title="意思地圖")
    fig.update_traces(textposition="top center", marker_size=12)
    fig.update_layout(width=750, height=550, xaxis_title="", yaxis_title="")
    fig.show()

## 記錄表（點兩下這格就能編輯）

**任務 1　詞向量遊戲**

| 題目 | 我猜的答案 | 模型的第一名 | 合理嗎？ |
|---|---|---|---|
| man : king = woman : ? | queen | queen | 合理 |
| france : paris = japan : ? | tokyo | tokyo | 合理 |
| walk : walked = swim : ? | swam | swam | 合理 |
| man : doctor = woman : ? | nurse | nurse | 合理 |
| 自己出的題目： | fruit | bananas | 不合理，但與第二名fruits只差0.01 |

**任務 2　句子相似度**

| 組合 | 我猜的分數 | 實際分數 | 跟我想的一樣嗎？ |
|---|---|---|---|
| 換句話說 | 0.9 | 0.892 | 近似 |
| 否定 | 0.7 | 0.841 | 近似 |
| 同主題 | 0.4 | 0.517 | 差一點 |
| 無關 | 0.1 | 0.483 | 差多了 |
| 數字 | 0.9 | 0.974 | 近似 |
| 一詞多義 | 0.6 | 0.510 | 近似 |

**任務 3　語意搜尋**

| 問題 | 關鍵字搜尋找對了嗎？ | 向量搜尋找對了嗎？ |
|---|---|---|
| 上網連不上 | 錯了 | 對了 |
| 電腦壞了要找誰 | 對了 | 對了 |
| 肚子餓了可以去哪裡吃飯 | 錯了 | 對了 |
| 畢業要多少學分 | 對了 | 對了 |
| 自己出的問題： | 對了 | 對了 |

## 三個問題（各一兩句話）
1. 「我喜歡這門課」和「我不喜歡這門課」的相似度高還是低？A:偏高 為什麼會這樣？A:因為兩句話大部分的詞都相同
2. 什麼情況下關鍵字搜尋反而比向量搜尋好用？A:要找特定名稱、編號或精確的字詞時
3. 如果要幫系上做一個「問答機器人」，你會怎麼用今天學到的東西？A:把系上的常見問題做成向量，學生提問時用向量搜尋找出意思最接近的問題，再回答對應的答案。